# 🧄 MS-FCAF: Multi-Scale Frequency-Channel Attention Fusion

**Đề tài:** Phân loại tỏi (Garlic Classification)

## Novel Contributions:
1. **FA-SE** (Frequency-Aware Squeeze-Excitation) — GAP + FFT band descriptors for channel attention
2. **CSAF** (Cross-Scale Attention Fusion) — Multi-head cross-attention between 3 backbone scales
3. **3-Scale Feature Extraction** — block3 (fine) + block5 (mid) + block7 (semantic)
4. **Adaptive CB Focal Loss** — Dynamic per-class weight adaptation


In [ ]:
# ============================================================

In [ ]:
import os, csv, time, random, shutil, glob, gc
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf
import keras.ops as ops

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Layer, Input, Concatenate, Reshape, Multiply, Add, Activation,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

print("=" * 60)
print("  MS-FCAF: Multi-Scale Frequency-Channel Attention Fusion")
print("=" * 60)
print(f"  TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPUs: {len(gpus)}")
else:
    print("  ⚠️ No GPU!")
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("  Mixed Precision: mixed_float16")

In [ ]:
STRATEGY_KEY   = "ms_fcaf_3scale"
STRATEGY_LABEL = "EfficientNetB4 + MS-FCAF (3-Scale Freq-Channel Attention Fusion)"

DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2944/dataset_final_2944"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

INPUT_SHAPE     = (380, 380, 3)
BATCH_SIZE      = 32
EPOCHS          = 30
LR              = 1e-4
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
DROPOUT_RATE    = 0.3
PATIENCE        = 12

FEAT_DIM        = 256
FA_SE_REDUCTION = 8
FA_SE_N_BANDS   = 2
CSAF_N_HEADS    = 4

FOCAL_GAMMA     = 2.0
CB_BETA         = 0.9999
ADAPTIVE_TAU    = 0.3

N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

print(f"  Strategy : {STRATEGY_LABEL}")
print(f"  3 Scales : block3 + block5 + block7")
print(f"  FA-SE    : reduction={FA_SE_REDUCTION}, bands={FA_SE_N_BANDS}")
print(f"  CSAF     : heads={CSAF_N_HEADS}, feat_dim={FEAT_DIM}")
print(f"  Loss     : Adaptive CB Focal (γ={FOCAL_GAMMA}, τ={ADAPTIVE_TAU})")
print(f"  Runs     : {N_RUNS} × seeds {RANDOM_SEEDS}")

In [ ]:
def build_backbone_3scale(input_shape):
    """EfficientNetB4 with 3 scale outputs: block3 + block5 + block7."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)

    def find_layer(prefix, keywords):
        for kw in keywords:
            for layer in base.layers:
                if prefix in layer.name and kw in layer.name:
                    return layer
        return None

    block3_layer = find_layer('block3', ['add', 'project_bn'])
    block5_layer = find_layer('block5', ['add', 'project_bn'])
    if block3_layer is None:
        block3_layer = find_layer('block2', ['add', 'project_bn'])
    if block5_layer is None:
        block5_layer = find_layer('block4', ['add', 'project_bn'])

    outputs = [block3_layer.output, block5_layer.output, base.output]
    for i, (name, out) in enumerate(zip(['Fine','Mid','Semantic'], outputs)):
        print(f"    Scale {i+1} ({name}): {out.shape}")

    multi_model = Model(inputs=base.input, outputs=outputs, name='effnetb4_3scale')
    return multi_model, base


class FrequencyAwareSE(Layer):
    """FA-SE: Combines GAP + FFT band descriptors for channel attention.
    Novel: frequency-aware channel recalibration on spatial maps."""

    def __init__(self, reduction=8, n_bands=2, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.n_bands = n_bands

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 4)
        self.proj = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_proj')
        self.fc1 = Dense(r, activation='relu', use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, activation='sigmoid', use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.band_weights = self.add_weight(
            name='band_importance', shape=(self.n_bands,),
            initializer=tf.keras.initializers.Constant(1.0 / self.n_bands),
            trainable=True)
        H, W = input_shape[1], input_shape[2]
        if H is not None and W is not None:
            self._build_masks(H, W)
        super().build(input_shape)

    def _build_masks(self, H, W):
        fy = np.fft.fftfreq(H).reshape(-1, 1)
        fx = np.fft.fftfreq(W).reshape(1, -1)
        radius = np.sqrt(fy**2 + fx**2)
        edges = np.linspace(0, radius.max(), self.n_bands + 1)
        self.freq_masks = []
        for i in range(self.n_bands):
            m = ((radius >= edges[i]) & (radius < edges[i+1])).astype(np.float32)
            if m.sum() == 0:
                m = np.ones_like(m) / (H * W)
            self.freq_masks.append(tf.constant(m, dtype=tf.float32))

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        gap = tf.reduce_mean(x_f32, axis=[1, 2])

        x_t = tf.transpose(x_f32, [0, 3, 1, 2])
        x_fft = tf.signal.fft2d(tf.complex(x_t, tf.zeros_like(x_t)))
        mag = tf.math.log1p(tf.abs(x_fft))

        band_w = tf.nn.softmax(tf.cast(self.band_weights, tf.float32))
        freq_fused = tf.zeros_like(gap)
        for i in range(self.n_bands):
            masked = mag * tf.cast(self.freq_masks[i][tf.newaxis, tf.newaxis, :, :], tf.float32)
            desc = tf.reduce_mean(masked, axis=[2, 3])
            freq_fused = freq_fused + tf.cast(band_w[i], tf.float32) * desc

        descriptor = tf.concat([gap, freq_fused], axis=-1)
        descriptor = self.proj(descriptor)
        attn = self.fc1(descriptor)
        attn = self.fc2(attn)
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[-1]])
        out = x_f32 * attn
        return tf.cast(out, x.dtype)

    def compute_output_spec(self, x, training=False):
        import keras
        return keras.KerasTensor(x.shape, dtype=x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'n_bands': self.n_bands})
        return cfg


class CrossScaleAttentionFusion(Layer):
    """CSAF: Bi-directional cross-attention between N scale features.
    Novel: scales query each other via multi-head attention + gated fusion."""

    def __init__(self, feat_dim=256, n_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.feat_dim = feat_dim
        self.n_heads = n_heads

    def build(self, input_shape):
        self.n_scales = len(input_shape)
        self.projs = [Dense(self.feat_dim, use_bias=False, dtype='float32',
                            name=f'{self.name}_proj_{i}') for i in range(self.n_scales)]
        self.cross_attn = tf.keras.layers.MultiHeadAttention(
            num_heads=self.n_heads, key_dim=self.feat_dim // self.n_heads,
            dropout=0.1, dtype='float32', name=f'{self.name}_mha')
        self.gate_fc = Dense(self.n_scales, activation='softmax', dtype='float32',
                             name=f'{self.name}_gate')
        self.out_bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        super().build(input_shape)

    def call(self, inputs, training=False):
        projected = [tf.cast(self.projs[i](tf.cast(inputs[i], tf.float32)), tf.float32)
                     for i in range(self.n_scales)]
        tokens = tf.stack(projected, axis=1)  # (B, N_scales, D)

        attended = self.cross_attn(query=tokens, key=tokens, value=tokens,
                                   training=training)  # (B, N, D)
        attended = attended + tokens  # residual

        gate_input = tf.concat(projected, axis=-1)
        gate_w = self.gate_fc(gate_input)  # (B, N_scales)

        fused = tf.zeros_like(projected[0])
        for i in range(self.n_scales):
            fused = fused + gate_w[:, i:i+1] * attended[:, i, :]

        fused = self.out_bn(fused, training=training)
        return fused

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'feat_dim': self.feat_dim, 'n_heads': self.n_heads})
        return cfg


class CastToFloat32(Layer):
    def call(self, x):
        return ops.cast(x, 'float32')


def build_msfcaf_model(input_shape, num_classes, feat_dim=256,
                        se_reduction=8, n_bands=2, n_heads=4, dropout_rate=0.3):
    """Build MS-FCAF: 3-Scale + FA-SE + CSAF + Classification Head."""
    backbone_multi, backbone_base = build_backbone_3scale(input_shape)
    inputs = Input(shape=input_shape, name='input_image')
    fine_map, mid_map, semantic_map = backbone_multi(inputs)

    fine_att = FrequencyAwareSE(reduction=se_reduction, n_bands=n_bands, name='fase_fine')(fine_map)
    mid_att = FrequencyAwareSE(reduction=se_reduction, n_bands=n_bands, name='fase_mid')(mid_map)
    sem_att = FrequencyAwareSE(reduction=se_reduction, n_bands=n_bands, name='fase_semantic')(semantic_map)

    fine_feat = GlobalAveragePooling2D(name='gap_fine')(fine_att)
    mid_feat = GlobalAveragePooling2D(name='gap_mid')(mid_att)
    sem_feat = GlobalAveragePooling2D(name='gap_semantic')(sem_att)

    fused = CrossScaleAttentionFusion(feat_dim=feat_dim, n_heads=n_heads, name='csaf')(
        [fine_feat, mid_feat, sem_feat])

    x = BatchNormalization(name='head_bn')(fused)
    x = Dense(feat_dim, activation='relu', kernel_regularizer=l2(1e-4), name='head_dense')(x)
    x = Dropout(dropout_rate, name='head_dropout')(x)
    x = CastToFloat32(name='cast_f32')(x)
    predictions = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=inputs, outputs=predictions, name='MS_FCAF_Model')
    return model, backbone_base

print("✅ MS-FCAF Architecture defined")
print("   Novel 1: FA-SE (Frequency-Aware SE on 3 spatial scales)")
print("   Novel 2: CSAF (Cross-Scale Attention Fusion with MHA)")

In [ ]:
class AdaptiveClassBalancedFocalLoss(tf.keras.losses.Loss):
    """Adaptive CB Focal Loss: static CB + focal + dynamic recall-based adaptation."""

    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * num_classes
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor')
        print(f"  CB weights: {dict(zip(range(num_classes), weights.round(4)))}")

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        combined = self.static_weights * self.adaptive_factor
        combined = combined / tf.reduce_mean(combined)
        sample_w = tf.reduce_sum(y_true * combined, axis=-1)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


class AdaptiveWeightCallback(Callback):
    """Updates adaptive CB weights each epoch based on per-class val recall."""

    def __init__(self, loss_fn, val_ds, num_classes, class_names, tau=0.3, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn = loss_fn
        self.val_ds = val_ds
        self.num_classes = num_classes
        self.class_names = class_names
        self.tau = tau
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            per_class_recall[c] = (y_pred[mask] == c).mean() if mask.sum() > 0 else 1.0
        epsilon = 0.1
        target = (1.0 - per_class_recall) + epsilon
        current = self.loss_fn.adaptive_factor.numpy()
        new_factor = (1.0 - self.tau) * current + self.tau * target
        new_factor = new_factor / new_factor.mean()
        self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
        self.history.append({'epoch': epoch + 1,
                             'per_class_recall': per_class_recall.copy(),
                             'adaptive_factor': new_factor.copy()})
        print(f"  [AdaptiveCB] Recall: {dict(zip(self.class_names, [f'{r:.3f}' for r in per_class_recall]))}")

print("✅ Adaptive CB Focal Loss + Callback defined")

In [ ]:
efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.10, 0.10),
    tf.keras.layers.RandomBrightness(factor=0.15),
    tf.keras.layers.RandomContrast(factor=0.15),
], name='augmentation')


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        if not os.path.isdir(d):
            continue
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    samples_per_class = [train_lbl.count(i) for i in range(num_classes)]
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        samples_per_class=samples_per_class,
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    print(f"  Classes: {class_names}")
    print(f"  Samples/class: {samples_per_class}")
    return train_ds, val_ds, test_ds, meta

print("✅ Data pipeline defined")

In [ ]:
def apply_freeze_strategy(base_model, unfreeze_blocks):
    base_model.trainable = False
    for layer in base_model.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base_model.layers)} layers trainable")


CUSTOM_OBJECTS = {
    'FrequencyAwareSE': FrequencyAwareSE,
    'CrossScaleAttentionFusion': CrossScaleAttentionFusion,
    'CastToFloat32': CastToFloat32,
    'AdaptiveClassBalancedFocalLoss': AdaptiveClassBalancedFocalLoss,
}


def build_and_compile_model(num_classes, samples_per_class, steps_per_epoch):
    model, backbone_base = build_msfcaf_model(
        input_shape=INPUT_SHAPE, num_classes=num_classes,
        feat_dim=FEAT_DIM, se_reduction=FA_SE_REDUCTION,
        n_bands=FA_SE_N_BANDS, n_heads=CSAF_N_HEADS,
        dropout_rate=DROPOUT_RATE)

    apply_freeze_strategy(backbone_base, UNFREEZE_BLOCKS)

    loss_fn = AdaptiveClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes, gamma=FOCAL_GAMMA, beta=CB_BETA)

    total_steps = steps_per_epoch * EPOCHS
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR, decay_steps=total_steps, alpha=1e-6)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    print(f"  Model params: {model.count_params():,}")
    return model, loss_fn

print("✅ Model builder defined")

In [ ]:
for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "=" * 70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("=" * 70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    model, loss_fn = build_and_compile_model(
        meta.num_classes, meta.samples_per_class, steps_per_epoch)

    if run_idx == 0:
        print(f"\n  Architecture: 3-Scale FA-SE + CSAF(heads={CSAF_N_HEADS})")
        print(f"  Classifier: BN→Dense({FEAT_DIM})→Dropout({DROPOUT_RATE})→Softmax")
        print(f"  Loss: Adaptive CB Focal (γ={FOCAL_GAMMA}, τ={ADAPTIVE_TAU})")

    adaptive_cb = AdaptiveWeightCallback(
        loss_fn=loss_fn, val_ds=val_ds, num_classes=meta.num_classes,
        class_names=meta.class_names, tau=ADAPTIVE_TAU)

    callbacks = [
        adaptive_cb,
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks)

    # Save adaptive weight history
    wh_df = pd.DataFrame([{
        'epoch': h['epoch'],
        **{f'recall_{cn}': h['per_class_recall'][i] for i, cn in enumerate(meta.class_names)},
        **{f'factor_{cn}': h['adaptive_factor'][i] for i, cn in enumerate(meta.class_names)}
    } for h in adaptive_cb.history])
    wh_df.to_csv(os.path.join(RESULT_DIR, 'adaptive_weight_history.csv'), index=False)

    # Evaluate
    pred_probs = model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(
        y_true_run, y_pred_run, target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    # Save classification report
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=meta.class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (Acc={test_acc:.4f})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.show(); plt.close()

    # Learning curves + Adaptive factors
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    hist = history.history
    axes[0].plot(hist['loss'], label='Train'); axes[0].plot(hist['val_loss'], label='Val')
    axes[0].set_title('Loss (Adaptive CB Focal)'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist['accuracy'], label='Train'); axes[1].plot(hist['val_accuracy'], label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
    for cn in meta.class_names:
        axes[2].plot(wh_df['epoch'], wh_df[f'factor_{cn}'], label=cn)
    axes[2].set_title('Adaptive CB Factors'); axes[2].legend(); axes[2].grid(alpha=0.3)
    axes[2].axhline(1.0, color='black', ls='--', lw=1, alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curves.png'), dpi=300)
    plt.show(); plt.close()

    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train, 'n_val': meta.n_val, 'n_test': meta.n_test,
        'adaptive_history': adaptive_cb.history,
    })

    print(f"\n  ✅ Run {run_idx+1}: Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 70)
print(f" ALL {N_RUNS} RUNS COMPLETED")
print("=" * 70)

In [ ]:
accuracies  = [r['accuracy'] for r in all_runs_results]
precisions  = [r['precision'] for r in all_runs_results]
recalls     = [r['recall'] for r in all_runs_results]
f1_scores   = [r['f1_score'] for r in all_runs_results]
class_names = all_runs_results[0]['class_names']

print(f"\n{'=' * 60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'=' * 60}")
print(f"  Accuracy  : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  Precision : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"  Recall    : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"  F1-Score  : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Per run acc: {[f'{a:.4f}' for a in accuracies]}")

print("\n  ADDITIONAL METRICS:")
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"    Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

print("\n  PER-CLASS METRICS (mean ± std):")
per_class_stats = {}
for cn in class_names:
    p_vals = [r['per_class_metrics'][cn]['precision'] for r in all_runs_results]
    r_vals = [r['per_class_metrics'][cn]['recall'] for r in all_runs_results]
    f_vals = [r['per_class_metrics'][cn]['f1-score'] for r in all_runs_results]
    per_class_stats[cn] = {
        'precision': {'mean': np.mean(p_vals), 'std': np.std(p_vals)},
        'recall': {'mean': np.mean(r_vals), 'std': np.std(r_vals)},
        'f1': {'mean': np.mean(f_vals), 'std': np.std(f_vals)},
    }
    print(f"    {cn:<28} P={np.mean(p_vals):.4f}±{np.std(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}±{np.std(r_vals):.4f}  "
          f"F1={np.mean(f_vals):.4f}±{np.std(f_vals):.4f}")

# Save overall summary
overall_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'metric': m, 'mean': np.mean(v), 'std': np.std(v)
} for m, v in [('accuracy', accuracies), ('precision', precisions),
               ('recall', recalls), ('f1_score', f1_scores)]])
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, 'overall_metrics_summary.csv'), index=False)

summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'precision': r['precision'],
    'recall': r['recall'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv'), index=False)
print(f"\n  ✅ Summary saved")

In [ ]:
# --- 9.1: Per-Class Metrics Bar Chart with Error Bars ---
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, (metric, label) in enumerate([('precision','Precision'), ('recall','Recall'), ('f1','F1-Score')]):
    means = [per_class_stats[cn][metric]['mean'] for cn in class_names]
    stds = [per_class_stats[cn][metric]['std'] for cn in class_names]
    ax.bar(x + i*width, means, width, yerr=stds, label=label,
           color=colors[i], alpha=0.85, capsize=4, edgecolor='white')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title(f'Per-Class Metrics — {STRATEGY_LABEL}', fontweight='bold')
ax.set_xticks(x + width); ax.set_xticklabels(class_names, fontsize=9)
ax.set_ylim(0.7, 1.05); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'per_class_metrics.png'), dpi=300)
plt.show()

# --- 9.2: Aggregate Confusion Matrix (Normalized + Raw) ---
best_run = max(all_runs_results, key=lambda r: r['accuracy'])
cm = confusion_matrix(best_run['y_true'], best_run['y_pred'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names,
            yticklabels=class_names, cmap='Blues', ax=axes[0])
axes[0].set_title(f'Raw CM (Best Run — Acc={best_run["accuracy"]:.4f})')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.3f', xticklabels=class_names,
            yticklabels=class_names, cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Normalized CM')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'agg_confusion_matrix.png'), dpi=300)
plt.show()

# --- 9.3: ROC Curves ---
num_classes = len(class_names)
y_true_bin = label_binarize(best_run['y_true'], classes=list(range(num_classes)))
pred_probs = best_run['pred_probs']

fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']
for i, cn in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors_roc[i % len(colors_roc)],
            lw=2, label=f'{cn} (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {STRATEGY_LABEL}', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300)
plt.show()

# --- 9.4: Adaptive Weight Evolution ---
n_runs_plot = len(all_runs_results)
fig, axes = plt.subplots(n_runs_plot, 2, figsize=(14, 5 * n_runs_plot))
if n_runs_plot == 1:
    axes = axes[np.newaxis, :]
for ridx, r in enumerate(all_runs_results):
    ah = r['adaptive_history']
    epochs_list = [h['epoch'] for h in ah]
    for ci, cn in enumerate(r['class_names']):
        recall_vals = [h['per_class_recall'][ci] for h in ah]
        axes[ridx, 0].plot(epochs_list, recall_vals, marker='o', ms=3, label=cn)
        factor_vals = [h['adaptive_factor'][ci] for h in ah]
        axes[ridx, 1].plot(epochs_list, factor_vals, marker='s', ms=3, label=cn)
    axes[ridx, 0].set_title(f'Val Recall (Run {r["run"]})'); axes[ridx, 0].legend()
    axes[ridx, 0].grid(alpha=0.3); axes[ridx, 0].set_ylim([0, 1.05])
    axes[ridx, 1].set_title(f'Adaptive Factors (Run {r["run"]})'); axes[ridx, 1].legend()
    axes[ridx, 1].grid(alpha=0.3); axes[ridx, 1].axhline(1.0, color='k', ls='--', lw=1, alpha=0.5)
plt.suptitle('Adaptive CB Loss — Weight Evolution', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'adaptive_weight_evolution.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✅ All thesis visualizations saved")

In [ ]:
gc.collect()
print("Computing t-SNE embeddings (best run)...")

# Rebuild model for feature extraction
best_model_path = os.path.join(best_run['result_dir'], 'best_model.keras')
if os.path.exists(best_model_path):
    feat_model = load_model(best_model_path, custom_objects=CUSTOM_OBJECTS)
    # Get features from head_dense layer (before dropout)
    try:
        feat_extractor = Model(inputs=feat_model.input,
                               outputs=feat_model.get_layer('head_dense').output,
                               name='feature_extractor')
    except:
        feat_extractor = Model(inputs=feat_model.input,
                               outputs=feat_model.layers[-3].output,
                               name='feature_extractor')

    # Rebuild test dataset
    random.seed(best_run['seed']); np.random.seed(best_run['seed']); tf.random.set_seed(best_run['seed'])
    _, _, test_ds_tsne, meta_tsne = create_tf_datasets(DATA_DIR, INPUT_SHAPE, 4, seed=best_run['seed'])

    features = feat_extractor.predict(test_ds_tsne, verbose=1, batch_size=4)
    y_true_tsne = meta_tsne.test_classes

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(y_true_tsne)-1))
    features_2d = tsne.fit_transform(features)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_tsne = ['#e74c3c', '#2ecc71', '#3498db']
    for i, cn in enumerate(meta_tsne.class_names):
        mask = y_true_tsne == i
        ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
                   c=colors_tsne[i % len(colors_tsne)], label=cn, alpha=0.7, s=40, edgecolors='white', lw=0.5)
    ax.set_title(f't-SNE — {STRATEGY_LABEL}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10, markerscale=1.5); ax.grid(alpha=0.2)
    ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_RESULT_DIR, 'tsne.png'), dpi=300)
    plt.show()
    print("✅ t-SNE saved")

    del feat_model, feat_extractor, features
    tf.keras.backend.clear_session()
    gc.collect()
else:
    print("⚠️ Best model not found, skipping t-SNE")

In [ ]:
# Save comprehensive report
report_lines = [
    "=" * 80,
    "MS-FCAF: MULTI-SCALE FREQUENCY-CHANNEL ATTENTION FUSION — EXPERIMENT REPORT",
    "=" * 80,
    f"Strategy: {STRATEGY_LABEL}",
    f"Key: {STRATEGY_KEY}",
    "",
    "NOVEL CONTRIBUTIONS:",
    "  1. FA-SE (Frequency-Aware Squeeze-Excitation)",
    "     - Combines GAP + FFT band descriptors for channel attention",
    "     - Applied on 3 spatial scales (block3 + block5 + block7)",
    "  2. CSAF (Cross-Scale Attention Fusion)",
    "     - Multi-head cross-attention between 3 scale features",
    "     - Gated combination with residual connections",
    "  3. Adaptive CB Focal Loss (inherited from baseline)",
    "     - Dynamic per-class weight adaptation based on val recall",
    "",
    f"Architecture: EfficientNetB4 + FA-SE(3 scales) + CSAF(heads={CSAF_N_HEADS})",
    f"  FA-SE: reduction={FA_SE_REDUCTION}, n_bands={FA_SE_N_BANDS}",
    f"  CSAF: feat_dim={FEAT_DIM}, n_heads={CSAF_N_HEADS}",
    f"  Unfreeze blocks: {UNFREEZE_BLOCKS}",
    f"  LR: {LR} (CosineDecay → 1e-6)",
    f"  Loss: Adaptive CB Focal (γ={FOCAL_GAMMA}, β={CB_BETA}, τ={ADAPTIVE_TAU})",
    "",
    f"Dataset: {DATA_DIR}",
    f"Runs: {N_RUNS} seeds = {RANDOM_SEEDS[:N_RUNS]}",
    "",
    "OVERALL PERFORMANCE (Mean ± Std):",
    "-" * 60,
    f"  Accuracy  : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}",
    f"  Precision : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}",
    f"  Recall    : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}",
    f"  F1-Score  : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
    "",
    "PER-CLASS F1-SCORE:",
]
for cn in class_names:
    s = per_class_stats[cn]['f1']
    report_lines.append(f"  {cn:<30} {s['mean']:.4f} ± {s['std']:.4f}")

report_lines.extend([
    "",
    "ADDITIONAL METRICS (per run):",
])
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    report_lines.append(f"  Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

report_text = "\n".join(report_lines)
print(report_text)
with open(os.path.join(BASE_RESULT_DIR, 'EXPERIMENT_REPORT.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

# ZIP all results
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024*1024)
print(f"\n✅ Archive: {zip_path}.zip ({zip_size:.2f} MB)")
print("DONE! 🎉")